# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

Simple words, honest numbers. Everything below is **observed** on one sampled month (March 2026) and is **directional**, for decision-support only.

**Plan:** (1) read two paper findings the way I want my own work to be read, (2) re-run my Week-5 model under a naive split and under an honest split and show both numbers, plus real failure examples, (3) hunt leakage in my final feature set, (4) rewrite my boldest claim in safe language.

## 1. Two paper findings + my methodology questions

*The aim is to practice the next level of rigor, not to grade the paper. I read FlyRank's "The State of AI-Driven SEO" (March 2026). These are the questions I would want a reviewer to ask me about my own work.*

**What the paper does well (so my questions start from respect):** it says its study is observational, it states that no p-values or confidence intervals are reported, it demotes weak results (it explains why the 361+ freshness ratio of 283:1 is unstable), it keeps reversed and nuanced findings visible, and it labels the machine-learning pages as exploratory and descriptive.

For each finding I ask the same two questions:
1. **Where does the label come from?** Is it measured directly, or is it a proxy built from other columns? Could the inputs and the label share a source?
2. **Does the validation design carry the claim?** What was held out (rows, pages, brands, time)? Does the split match the sentence the claim makes?

### Finding A: The Freshness Multiplier (Finding #4)
- **The finding:** pages older than 365 days that were refreshed within the last 30 days show about 3.2x the health score (10.7 to 34.5) and about 57x the impressions (71 to 4,039). The paper calls refresh timing "one of the strongest measured levers available".
- **Where does the label come from?** There are two outcomes. Impressions is a direct Search Console count. Health score is a FlyRank composite (impressions 30 points, position 30, CTR 20, scroll depth 20), so it is partly built from the same quantity as the impressions outcome, and the paper itself says it is a summary metric, not a market-standard outcome. The extended cuts also use an active-content subset (impressions above 0 and sessions above 0), so stale pages that earned nothing are not in the comparison. The paper names this survivor effect for the small 361+ cell; the same filter could also shape the 71 versus 4,039 comparison.
- **Does the validation design carry the claim?** The comparison reads as refreshed pages versus unrefreshed pages, with no before/after on the same pages, no matching, and no group sizes beside the numbers. Impressions are very skewed (the paper's own table shows a median of 668 against a mean of about 3,510), so a mean ratio like 57x can be driven by a few large pages. How pages were chosen for refresh is not stated; if refreshes went to pages already gaining traffic, part of the gap could be selection rather than the refresh.
- **My constructive question to the authors:** could the refresh result be shown as a before/after on the same pages, or against unrefreshed pages with similar impressions before the refresh date, with group sizes and medians next to the means?
- **What would raise my confidence:** a matched comparison on prior-period impressions and age, reported with n and medians, plus a sentence on how pages were selected for refresh. I would then be comfortable reading it as a strong association that supports refreshing as a decision-support priority.

### Finding B: What Predicts Growth (ML appendix, logistic regression)
- **The finding:** a logistic regression separates growing from declining pages with 71% holdout accuracy on an 80/20 split. Content age is the strongest negative signal, and days visible and recent impressions are among the strongest positive signals.
- **Where does the label come from?** Growing and declining come from the 30-day versus previous-30-day impression change (more than 10% up or down). The features include impressions, clicks and days visible, and the text reads "recent impressions" as a positive signal. When inputs and the label come from the same impressions series, some of the accuracy can be the label showing through. A percentage change on a small base also swings easily.
- **Does the validation design carry the claim?** The methodology lists a random 80/20 split. It does not say whether the 57 brands were held out as groups or whether features come from a period before the label window, so pages from one brand can sit on both sides of the split. The 71% is also reported without a majority-class baseline: if the class mix resembles Finding #1 (74.8K rising versus 45.6K falling), always guessing "growing" would already score about 62%. This is my assumption, since the modeling sample's class mix is not shown.
- **My constructive question to the authors:** could the 71% be shown next to a majority-class baseline, and again under a split that holds out whole brands or a later time window, with features measured before the label window?
- **What would raise my confidence:** the same model under a brand-grouped, time-forward split. A small accuracy drop would support the descriptive reading the paper already asks for, and a large drop would show how much came from sharing brands or sharing the impressions series.

### How this connects to my own work
In Weeks 5 and 6 I hold out whole clients, build features from days 1-21 and the label from days 22-31, and report a baseline next to every model. The two questions above are the same ones Section 2 and Section 3 apply to my model.

### Tone checklist (keep these true)
- I name what the paper does well first.
- My questions are concrete and answerable, not "is this wrong?".
- I do not say "the paper is flawed". I say "the claim would be easier to carry if ...".

In [14]:
# Short versions of the Section 1 answers. This cell also checks that nothing is left as a placeholder.
paper_findings = {
    "A": {
        "finding": "Finding #4 Freshness Multiplier: pages older than 365 days refreshed within 30 days show about 3.2x health (10.7 to 34.5) and 57x impressions (71 to 4,039).",
        "label_origin": "Impressions is a direct count; health is a FlyRank composite partly built from impressions, position, CTR and scroll depth. The active-content filter (impressions and sessions above 0) drops stale pages that earned nothing.",
        "validation_question": "It reads as a between-group comparison with no before/after on the same pages, no matching, no group sizes and mean-based ratios on skewed impressions. Refresh selection is not described.",
        "constructive_question": "Could the refresh result be shown as a before/after on the same pages, or against unrefreshed pages with similar prior impressions, with n and medians beside the means?",
    },
    "B": {
        "finding": "ML appendix: logistic regression separates growing from declining pages with 71% holdout accuracy on an 80/20 split.",
        "label_origin": "Growing and declining come from the 30-day versus previous-30-day impression change, while the features include impressions, clicks and days visible, so inputs and label may share a source.",
        "validation_question": "A random 80/20 split is listed; brand-grouped or time-forward holdout is not stated, and no majority-class baseline is shown (about 62% if the class mix resembles Finding #1, my assumption).",
        "constructive_question": "Could the 71% be reported next to a majority-class baseline and under a split that holds out whole brands or a later time window, with features measured before the label window?",
    },
}
left = [(k, f) for k, v in paper_findings.items() for f, t in v.items() if "[" in t and "]" in t]
if left:
    print("WARNING: Section 1 still has placeholders:", left)
else:
    print("Section 1 filled: findings", list(paper_findings))

Section 1 filled: findings ['A', 'B']


## 2. My model under an honest split (before/after)

*Re-run the Week-5 model under a naive random-row split (BEFORE) and under a grouped-by-client split (AFTER). Same data, same features, same label rule, same metrics. Only the split changes.*

- **BEFORE:** random rows. Pages of the same client land in both train and test, so the model can lean on client habits it already saw.
- **AFTER:** `GroupKFold` by client. Test clients are never seen in training. Features come from days 1-21 and the label from days 22-31, so it is also time-forward.

In [15]:
import os, json
import numpy as np
import pandas as pd
from huggingface_hub import login

# Token comes from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Add it in Colab -> Secrets and enable notebook access.")

os.environ["HF_TOKEN"] = token
login(token=token)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [16]:
!pip install -q huggingface_hub pandas pyarrow scikit-learn

In [17]:
storage_options = {"token": token}
BASE = "hf://datasets/FlyRank/internship-warehouse"

df_raw = pd.read_parquet(f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options=storage_options)
df_content = pd.read_parquet(f"{BASE}/dim_content.parquet", storage_options=storage_options)
print("Performance rows:", len(df_raw), "| content rows:", len(df_content))

Performance rows: 9841378 | content rows: 519606


In [18]:

KEY = ["client_hash_id", "content_hash_id"]
need = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
        "gsc_sum_position", "client_has_gsc", "gsc_data_available"]
missing_cols = [c for c in need if c not in df_raw.columns]
assert not missing_cols, f"Missing columns in fact table: {missing_cols} -> rename them in this cell"

df = df_raw[(df_raw["client_has_gsc"] == True) & (df_raw["gsc_data_available"] == True)].copy()
df["report_date"] = pd.to_datetime(df["report_date"])
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    df[c] = df[c].fillna(0)
df["active"] = (df["gsc_impressions"] > 0).astype(int)

START = df["report_date"].min()
CUTOFF = START + pd.Timedelta(days=20)          # features: first 21 days
obs = df[df["report_date"] <= CUTOFF].copy()
later = df[df["report_date"] > CUTOFF].copy()   # label only, never a feature
print("Feature window:", obs["report_date"].min().date(), "to", obs["report_date"].max().date())
print("Label window:  ", later["report_date"].min().date(), "to", later["report_date"].max().date())

def agg_pages(d):
    return d.groupby(KEY).agg(
        impressions=("gsc_impressions", "sum"), clicks=("gsc_clicks", "sum"),
        sum_pos=("gsc_sum_position", "sum"), days_active=("active", "sum"),
        max_day_impr=("gsc_impressions", "max")).reset_index()

pages = agg_pages(obs)
pages = pages[pages["impressions"] > 0].copy()
pages["ctr"] = pages["clicks"] / pages["impressions"]
raw_pos = pages["sum_pos"] / pages["impressions"]
POSITION_OFFSET = 1 if raw_pos.min() < 1 else 0
pages["avg_position"] = raw_pos + POSITION_OFFSET
pages["spike_share"] = pages["max_day_impr"] / pages["impressions"]

MIN_IMPR, MAX_POS, MIN_POS = 100, 20, 1
RULE_BINS = [0, 1.5, 2.5, 3.5, 5, 7, 10, 15, 20]
RULE_LABELS = ["<=1.5", "1.5-2.5", "2.5-3.5", "3.5-5", "5-7", "7-10", "10-15", "15-20"]
pages["eligible"] = (pages["impressions"] >= MIN_IMPR) & (pages["avg_position"] >= MIN_POS) & (pages["avg_position"] <= MAX_POS)
pages["rule_bin"] = pd.cut(pages["avg_position"], bins=RULE_BINS, labels=RULE_LABELS)
norm = (pages[pages["eligible"]].groupby("rule_bin", observed=True)
        .agg(impressions=("impressions", "sum"), clicks=("clicks", "sum")).reset_index())
norm["expected_ctr"] = norm["clicks"] / norm["impressions"]
pages = pages.merge(norm[["rule_bin", "expected_ctr"]], on="rule_bin", how="left")

# ---- Features (first 21 days + static content only)
d = obs.copy()
d["pos_day"] = d["gsc_sum_position"] / d["gsc_impressions"].where(d["gsc_impressions"] > 0) + POSITION_OFFSET
pos_std = d.groupby(KEY)["pos_day"].std().rename("pos_std").reset_index()
l7 = d[d["report_date"] > CUTOFF - pd.Timedelta(days=7)]
l7 = l7.groupby(KEY).agg(impr_l7=("gsc_impressions", "sum"), clicks_l7=("gsc_clicks", "sum")).reset_index()

M = pages[pages["eligible"]].copy()
M = M.merge(pos_std, on=KEY, how="left").merge(l7, on=KEY, how="left")
M["pos_std"] = M["pos_std"].fillna(0)
M[["impr_l7", "clicks_l7"]] = M[["impr_l7", "clicks_l7"]].fillna(0)
M["impr_l7_share"] = M["impr_l7"] / M["impressions"]
M["ctr_l7"] = np.where(M["impr_l7"] > 0, M["clicks_l7"] / M["impr_l7"].where(M["impr_l7"] > 0), M["ctr"])

cont = df_content[["content_hash_id", "search_volume", "word_count", "char_count", "competition_level"]].drop_duplicates("content_hash_id")
M = M.merge(cont, on="content_hash_id", how="left")
for c in ["search_volume", "word_count", "char_count"]:
    M[f"{c}_missing"] = M[c].isna().astype(int)
M["comp_missing"] = M["competition_level"].isna().astype(int)
M["competition"] = M["competition_level"].astype(str).str.upper().map({"LOW": 1, "MEDIUM": 2, "HIGH": 3}).fillna(0)
M["log_impr"] = np.log1p(M["impressions"])
M["log_vol"] = np.log1p(M["search_volume"].fillna(0))
M["log_words"] = np.log1p(M["word_count"].fillna(0))
M["log_chars"] = np.log1p(M["char_count"].fillna(0))

FEATURES = ["log_impr", "ctr", "avg_position", "days_active", "spike_share", "pos_std",
            "impr_l7_share", "ctr_l7", "log_vol", "log_words", "log_chars", "competition",
            "search_volume_missing", "word_count_missing", "char_count_missing", "comp_missing"]

lp = agg_pages(later)[KEY + ["impressions", "clicks"]].rename(columns={"impressions": "later_impr", "clicks": "later_clicks"})
M = M.merge(lp, on=KEY, how="left")
M[["later_impr", "later_clicks"]] = M[["later_impr", "later_clicks"]].fillna(0)
M = M.reset_index(drop=True)
print("Modeling pages:", len(M), "| clients:", M["client_hash_id"].nunique(), "| features:", len(FEATURES))

Feature window: 2026-03-01 to 2026-03-21
Label window:   2026-03-22 to 2026-03-31
Modeling pages: 68868 | clients: 38 | features: 16


In [19]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

TOP_FRAC = 0.05
n_clients = M["client_hash_id"].nunique()
n_splits = min(5, n_clients)
assert n_splits >= 3, f"Only {n_clients} clients: grouped validation needs at least 3"
MODEL_NAMES = ["Logistic Regression", "Random Forest (shallow)", "Gradient Boosting (small)"]

def make_models():
    return {
        "Logistic Regression": Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))]),
        "Random Forest (shallow)": RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, n_jobs=-1, random_state=42),
        "Gradient Boosting (small)": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=42),
    }

def metrics(y, score, missed):
    y = np.asarray(y); score = np.asarray(score, float); missed = np.asarray(missed, float)
    k = max(10, int(np.ceil(TOP_FRAC * len(y))))
    top = np.argsort(-score, kind="stable")[:k]
    return {"AUC": roc_auc_score(y, score), "AvgPrecision": average_precision_score(y, score),
            "Precision@top5%": y[top].mean(), "Captured@top5%": missed[top].sum() / max(missed.sum(), 1e-9)}

group_splits = list(GroupKFold(n_splits=n_splits).split(M, groups=M["client_hash_id"]))
random_splits = list(KFold(n_splits=n_splits, shuffle=True, random_state=42).split(M))

def fold_labels(tr, norm_mode="pooled"):
    # expected CTR: pooled (as in Weeks 4-5) or learned from the TRAIN rows of this fold only
    if norm_mode == "pooled":
        exp = M["expected_ctr"].to_numpy()
    else:
        t = M.iloc[tr]
        g = t.groupby("rule_bin", observed=True).agg(i=("impressions", "sum"), c=("clicks", "sum"))
        cmap = (g["c"] / g["i"]); cmap.index = cmap.index.astype(str)
        exp = M["rule_bin"].astype(str).map(cmap).astype(float).fillna(t["clicks"].sum() / t["impressions"].sum()).to_numpy()
    missed = np.clip(exp * M["later_impr"].to_numpy() - M["later_clicks"].to_numpy(), 0, None)
    base = np.clip(exp * M["impressions"].to_numpy() - M["clicks"].to_numpy(), 0, None)
    thr = max(float(np.quantile(missed[tr], 0.75)), 1.0)      # threshold from TRAIN pages only
    return (missed >= thr).astype(int), missed, base

def run_cv(splits, norm_mode="pooled", features=None, models=True):
    features = features or FEATURES
    rows, oof = [], {n: np.full(len(M), np.nan) for n in MODEL_NAMES}
    fold_y, fold_missed, fold_te = {}, {}, {}
    for fold, (tr, te) in enumerate(splits):
        y_all, missed, base = fold_labels(tr, norm_mode)
        ytr, yte = y_all[tr], y_all[te]
        fold_y[fold], fold_missed[fold], fold_te[fold] = y_all, missed, te
        if len(set(ytr)) < 2 or len(set(yte)) < 2:
            continue
        refs = {"Week-4 rule score (baseline)": base[te], "Impressions only": M["log_impr"].to_numpy()[te]}
        for nm, sc in refs.items():
            rows.append({"fold": fold, "model": nm, **metrics(yte, sc, missed[te])})
        if models:
            for nm, mdl in make_models().items():
                mdl.fit(M.iloc[tr][features], ytr)
                sc = mdl.predict_proba(M.iloc[te][features])[:, 1]
                oof[nm][te] = sc
                rows.append({"fold": fold, "model": nm, **metrics(yte, sc, missed[te])})
    return {"res": pd.DataFrame(rows), "oof": oof, "fold_y": fold_y, "fold_missed": fold_missed, "fold_te": fold_te}

print("Running BEFORE (random rows) ...")
R_random = run_cv(random_splits, "pooled")
print("Running AFTER (grouped by client) ...")
R_group = run_cv(group_splits, "pooled")
print("Done. Folds:", n_splits, "| clients:", n_clients)

Running BEFORE (random rows) ...
Running AFTER (grouped by client) ...
Done. Folds: 5 | clients: 38


In [20]:
cols = ["AUC", "AvgPrecision", "Precision@top5%", "Captured@top5%"]
order = ["Week-4 rule score (baseline)", "Impressions only"] + MODEL_NAMES

def summarize(res):
    return res.groupby("model")[cols].mean().loc[[m for m in order if m in res["model"].unique()]]

s_before, s_after = summarize(R_random["res"]), summarize(R_group["res"])
cmp = pd.concat({"BEFORE (random rows)": s_before, "AFTER (grouped by client)": s_after}, axis=1).round(3)
print("MEAN METRICS OVER FOLDS")
print(cmp.to_string())

gap = (s_before[["AUC", "Captured@top5%"]] - s_after[["AUC", "Captured@top5%"]]).round(3)
gap.columns = ["AUC_inflation", "Captured_inflation"]
print("\nINFLATION = BEFORE minus AFTER (positive means the naive split looked better than the honest one)")
print(gap.to_string())

best_after = s_after.loc[MODEL_NAMES, "AUC"].idxmax()
print(f"\nBest model under the honest split: {best_after}")
print(f"  AUC before {s_before.loc[best_after,'AUC']:.3f} -> after {s_after.loc[best_after,'AUC']:.3f}")
print(f"  Week-4 rule AUC (no training, so it barely changes): {s_before.loc['Week-4 rule score (baseline)','AUC']:.3f} -> {s_after.loc['Week-4 rule score (baseline)','AUC']:.3f}")
print("Reading (directional): inflation shows how much a naive split would have flattered a model that learns from rows it has effectively seen.")

MEAN METRICS OVER FOLDS
                             BEFORE (random rows)                                             AFTER (grouped by client)                                            
                                              AUC AvgPrecision Precision@top5% Captured@top5%                       AUC AvgPrecision Precision@top5% Captured@top5%
model                                                                                                                                                              
Week-4 rule score (baseline)                0.795        0.695           0.940          0.411                     0.785        0.668           0.904          0.388
Impressions only                            0.844        0.591           0.695          0.364                     0.837        0.552           0.635          0.338
Logistic Regression                         0.885        0.729           0.877          0.421                     0.876        0.687           0.830        

### Real failure examples under the honest split
*Pages the best model put in its review queue (top 5% per fold) that did not stay a big prize (false alarms), and big prizes it missed. Ids are shortened hashes; no client names or URLs.*

In [21]:
oof_best = R_group["oof"][best_after]
Z = M.copy()
Z["score"] = oof_best
Z["fold"] = np.nan; Z["y"] = np.nan; Z["later_missed_f"] = np.nan; Z["picked"] = False
for f, te in R_group["fold_te"].items():
    Z.iloc[te, Z.columns.get_loc("fold")] = f
    Z.iloc[te, Z.columns.get_loc("y")] = R_group["fold_y"][f][te]
    Z.iloc[te, Z.columns.get_loc("later_missed_f")] = R_group["fold_missed"][f][te]
    sc = Z.iloc[te]["score"]
    if sc.isna().all(): continue
    k = max(10, int(np.ceil(TOP_FRAC * len(te))))
    Z.loc[sc.sort_values(ascending=False).index[:k], "picked"] = True
Z = Z[Z["y"].notna()].copy()
Z["score_pct"] = Z.groupby("fold")["score"].rank(pct=True)

def likely_reason(r):
    if r.spike_share >= 0.30: return "one-day spike inflated the first window"
    if r.days_active < 10: return "thin history (few active days)"
    if r.impr_l7_share < 0.15: return "impressions faded in the last observed week"
    if r.impr_l7_share > 0.50: return "impressions surged late (recent change)"
    return "no obvious pattern in the inputs"

show = ["content", "impressions", "avg_position", "ctr", "days_active", "spike_share", "later_missed_f", "score_pct", "likely_reason"]
Z["content"] = Z["content_hash_id"].astype(str).str[-6:]

fa = Z[Z["picked"] & (Z["y"] == 0)].sort_values("score", ascending=False).head(8).copy()
fa["likely_reason"] = fa.apply(likely_reason, axis=1)
mi = Z[(~Z["picked"]) & (Z["y"] == 1)].sort_values("later_missed_f", ascending=False).head(8).copy()
mi["likely_reason"] = mi.apply(likely_reason, axis=1)

print("FALSE ALARMS (model said 'big prize', later gap was small):")
print(fa[show].round(3).to_string(index=False))
print("\nBIGGEST MISSES (real prize the model ranked low):")
print(mi[show].round(3).to_string(index=False))

n_pick = int(Z["picked"].sum())
print(f"\nQueue precision: {float((Z[Z['picked']]['y']==1).mean()):.3f} on {n_pick} picked pages | base rate {float(Z['y'].mean()):.3f}")
print("False-alarm reasons (top 8):", fa["likely_reason"].value_counts().to_dict())
print("Miss reasons (top 8):", mi["likely_reason"].value_counts().to_dict())

FALSE ALARMS (model said 'big prize', later gap was small):
content  impressions  avg_position   ctr  days_active  spike_share  later_missed_f  score_pct                           likely_reason
 63ab80        14790         3.929 0.000           21        0.412           0.000      0.996 one-day spike inflated the first window
 8a4379         7231         3.026 0.001           21        0.354           0.000      0.987 one-day spike inflated the first window
 5cc15d        28861         8.074 0.000           21        0.127           0.018      0.995        no obvious pattern in the inputs
 c3f61e         8870         5.159 0.001           21        0.385           0.000      0.984 one-day spike inflated the first window
 ed140e         6214         3.236 0.000           21        0.247           0.046      0.995 impressions surged late (recent change)
 b0f272         4722         4.913 0.001           21        0.076           0.000      0.977        no obvious pattern in the inputs
 7

## 3. Leakage audit

*The same hunt as Week 3, on my final feature set. Each check prints PASS or FLAG so the result is read from numbers, not feelings.*

**Checks:** (a) where every feature comes from and when it is known, (b) banned-name and window guards, (c) single-feature scan with a random-noise control, (d) a deliberate label-derived trap to show what a leak looks like, (e) overlap of pages and content across train and test, (f) the pooled-norm leak, measured by re-learning the CTR norm from training clients only.

In [22]:
# (a) Provenance table: every feature, where it comes from, when it is known
provenance = {
    "log_impr": ("impressions summed over days 1-21", "day 21"),
    "ctr": ("clicks / impressions over days 1-21", "day 21"),
    "avg_position": ("position sums / impressions, days 1-21", "day 21"),
    "days_active": ("count of days with impressions, days 1-21", "day 21"),
    "spike_share": ("max single-day impressions / total, days 1-21", "day 21"),
    "pos_std": ("std of daily position, days 1-21", "day 21"),
    "impr_l7_share": ("impressions in days 15-21 / days 1-21", "day 21"),
    "ctr_l7": ("CTR over days 15-21", "day 21"),
    "log_vol": ("keyword search volume (dim_content)", "before publishing"),
    "log_words": ("word count (dim_content)", "before publishing"),
    "log_chars": ("character count (dim_content)", "before publishing"),
    "competition": ("keyword competition level (dim_content)", "before publishing"),
    "search_volume_missing": ("flag: volume missing", "static"),
    "word_count_missing": ("flag: word count missing", "static"),
    "char_count_missing": ("flag: char count missing", "static"),
    "comp_missing": ("flag: competition missing", "static"),
}
assert set(provenance) == set(FEATURES), f"Provenance table out of sync: {set(FEATURES) ^ set(provenance)}"
prov = pd.DataFrame([{"feature": k, "source": v[0], "known_by": v[1]} for k, v in provenance.items()])
print(prov.to_string(index=False))

audit = []   # (check, status, detail)

# (b) Name and window guards
banned = ["paid", "ai_", "ga4_", "later", "label", "product", "flag", "baseline", "expected", "client_hash", "missed", "y_"]
bad = [f for f in FEATURES if any(b in f for b in banned)]
window_ok = bool(obs["report_date"].max() <= CUTOFF < later["report_date"].min())
audit.append(("Banned input names", "PASS" if not bad else "FLAG", str(bad) if bad else "none of: paid, AI, GA4, product flags, label, Week-4 score, client id"))
audit.append(("Feature window ends before label window", "PASS" if window_ok else "FLAG",
              f"features end {obs['report_date'].max().date()}, label starts {later['report_date'].min().date()}"))

              feature                                        source          known_by
             log_impr             impressions summed over days 1-21            day 21
                  ctr           clicks / impressions over days 1-21            day 21
         avg_position        position sums / impressions, days 1-21            day 21
          days_active     count of days with impressions, days 1-21            day 21
          spike_share max single-day impressions / total, days 1-21            day 21
              pos_std              std of daily position, days 1-21            day 21
        impr_l7_share         impressions in days 15-21 / days 1-21            day 21
               ctr_l7                           CTR over days 15-21            day 21
              log_vol           keyword search volume (dim_content) before publishing
            log_words                      word count (dim_content) before publishing
            log_chars                 character count 

In [23]:
# (c) Single-feature scan on held-out clients (with a random-noise control)
rng = np.random.default_rng(0)
M["noise_control"] = rng.normal(size=len(M))
scan = []
for feat in FEATURES + ["noise_control"]:
    aucs = []
    for f, te in R_group["fold_te"].items():
        yte = R_group["fold_y"][f][te]
        if len(set(yte)) < 2: continue
        a = roc_auc_score(yte, M[feat].to_numpy()[te]); aucs.append(max(a, 1 - a))   # direction-free
    scan.append({"feature": feat, "single_feature_AUC": float(np.mean(aucs))})
scan = pd.DataFrame(scan).sort_values("single_feature_AUC", ascending=False)
print(scan.round(3).to_string(index=False))
noise_auc = float(scan.loc[scan["feature"] == "noise_control", "single_feature_AUC"].iloc[0])
high = scan[(scan["single_feature_AUC"] > 0.85) & (scan["feature"] != "noise_control")]["feature"].tolist()
n_te_avg = float(np.mean([len(te) for te in R_group["fold_te"].values()]))
p_pos = float(np.mean([R_group["fold_y"][f][te].mean() for f, te in R_group["fold_te"].items()]))
se_null = float(np.sqrt((n_te_avg + 1) / (12 * n_te_avg * p_pos * n_te_avg * (1 - p_pos))))
noise_limit = 0.5 + max(0.03, 3 * se_null)       # a pure-noise column should stay inside this band
audit.append(("Random-noise control stays near 0.5", "PASS" if noise_auc < noise_limit else "FLAG", f"noise AUC {noise_auc:.3f} (limit {noise_limit:.3f})"))
audit.append(("No single feature above 0.85 AUC", "PASS" if not high else "FLAG",
              "none" if not high else f"{high}: high is not proof of a leak, but check provenance above (impressions legitimately predict size of prize)"))

# (d) Label-derived trap: what a leak looks like (NOT used in the model)
def grouped_auc(feats, extra=None):
    X = M[feats].copy()
    if extra is not None:
        for k, v in extra.items(): X[k] = v
    aucs = []
    for f, (tr, te) in enumerate(group_splits):
        y_all = R_group["fold_y"][f]
        if len(set(y_all[tr])) < 2 or len(set(y_all[te])) < 2: continue
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=100, random_state=42)
        m.fit(X.iloc[tr], y_all[tr]); aucs.append(roc_auc_score(y_all[te], m.predict_proba(X.iloc[te])[:, 1]))
    return float(np.mean(aucs))

auc_clean = grouped_auc(FEATURES)
trap_col = np.clip(M["expected_ctr"].to_numpy() * M["later_impr"].to_numpy() - M["later_clicks"].to_numpy(), 0, None)  # label source
auc_trap = grouped_auc(FEATURES, {"trap_label_derived": trap_col})
print(f"\nLabel-derived trap: clean AUC {auc_clean:.3f} -> with later missed clicks as a feature {auc_trap:.3f} (trap column dropped afterwards)")
audit.append(("Trap column visibly inflates AUC (so a real leak would show)", "PASS" if auc_trap > auc_clean + 0.05 else "FLAG",
              f"{auc_clean:.3f} -> {auc_trap:.3f}"))

              feature  single_feature_AUC
             log_impr               0.837
              pos_std               0.668
          spike_share               0.605
            log_words               0.585
         avg_position               0.576
            log_chars               0.572
   char_count_missing               0.566
   word_count_missing               0.566
        impr_l7_share               0.565
               ctr_l7               0.549
          days_active               0.549
              log_vol               0.548
                  ctr               0.546
          competition               0.513
        noise_control               0.509
         comp_missing               0.507
search_volume_missing               0.505

Label-derived trap: clean AUC 0.905 -> with later missed clicks as a feature 1.000 (trap column dropped afterwards)


In [24]:
# (e) Overlap of pages / content between train and test under the honest split
dup_pages = int(M.duplicated(subset=KEY).sum())
multi_client = int((M.groupby("content_hash_id")["client_hash_id"].nunique() > 1).sum())
shared = []
for tr, te in group_splits:
    shared.append(len(set(M.iloc[tr]["client_hash_id"]) & set(M.iloc[te]["client_hash_id"])))
audit.append(("One row per (client, content) page", "PASS" if dup_pages == 0 else "FLAG", f"duplicate pages: {dup_pages}"))
audit.append(("No client in both train and test", "PASS" if max(shared) == 0 else "FLAG", f"shared clients per fold: {shared}"))
audit.append(("Content ids shared across clients", "PASS" if multi_client == 0 else "NOTE", f"{multi_client} content ids appear under more than one client"))

# (f) The pooled-norm leak: re-learn the CTR norm from TRAIN clients only and redo the comparison
R_trainnorm = run_cv(group_splits, "train_only", models=False)
b_pooled = R_group["res"][R_group["res"]["model"] == "Week-4 rule score (baseline)"]["AUC"].mean()
b_train = R_trainnorm["res"][R_trainnorm["res"]["model"] == "Week-4 rule score (baseline)"]["AUC"].mean()
i_pooled = R_group["res"][R_group["res"]["model"] == "Impressions only"]["AUC"].mean()
i_train = R_trainnorm["res"][R_trainnorm["res"]["model"] == "Impressions only"]["AUC"].mean()
print(f"\nPooled norm vs train-only norm (label and rule both rebuilt per fold):")
print(f"  Week-4 rule AUC: pooled {b_pooled:.3f} -> train-only {b_train:.3f}")
print(f"  Impressions-only AUC: pooled {i_pooled:.3f} -> train-only {i_train:.3f}")
norm_shift = abs(b_pooled - b_train)
audit.append(("Pooled CTR norm does not change the baseline by more than 0.01 AUC", "PASS" if norm_shift <= 0.01 else "FLAG",
              f"shift {norm_shift:.3f}"))

A = pd.DataFrame(audit, columns=["check", "status", "detail"])
print("\nLEAKAGE AUDIT SUMMARY")
print(A.to_string(index=False))
print("\nOverall:", "no FLAGs" if not (A["status"] == "FLAG").any() else f"{int((A['status']=='FLAG').sum())} FLAG(s): read the detail column and explain each in your notes")


Pooled norm vs train-only norm (label and rule both rebuilt per fold):
  Week-4 rule AUC: pooled 0.785 -> train-only 0.786
  Impressions-only AUC: pooled 0.837 -> train-only 0.835

LEAKAGE AUDIT SUMMARY
                                                             check status                                                                detail
                                                Banned input names   PASS none of: paid, AI, GA4, product flags, label, Week-4 score, client id
                           Feature window ends before label window   PASS                      features end 2026-03-21, label starts 2026-03-22
                               Random-noise control stays near 0.5   PASS                                         noise AUC 0.509 (limit 0.530)
                                  No single feature above 0.85 AUC   PASS                                                                  none
      Trap column visibly inflates AUC (so a real leak would show)   PASS   

### Reading the audit
- A **PASS** means the check did not find a problem in this sample. It does not prove the absence of leakage.
- The **trap column** is deliberately leaky and is never part of the model. It is here to show that the audit would notice a real label-derived leak.
- A high single-feature AUC is a prompt to check provenance, not a verdict. Impressions legitimately relate to the size of a prize.
- The **pooled norm** is a disclosed, small design choice from Week 4. The train-only re-run measures how much it matters.
- Limits that remain: one month, one label definition (a proxy built from the same page's later CTR gap), and page-level data only.

## 4. Claim rewrite

*Take my own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [25]:
# Numbers come from the runs above, so the rewritten claim cannot drift from the evidence.
res_g = R_group["res"]
base_f = res_g[res_g["model"] == "Week-4 rule score (baseline)"].set_index("fold")["AUC"]
mf = res_g[res_g["model"] == best_after].set_index("fold")["AUC"]
common = base_f.index.intersection(mf.index)
wins = int((mf.loc[common] > base_f.loc[common]).sum()); nf = len(common)
auc_model, auc_base = float(mf.loc[common].mean()), float(base_f.loc[common].mean())
auc_naive = float(s_before.loc[best_after, "AUC"])
imp_auc = float(s_after.loc["Impressions only", "AUC"])
cap_model = float(s_after.loc[best_after, "Captured@top5%"]); cap_base = float(s_after.loc["Week-4 rule score (baseline)", "Captured@top5%"])
gain = auc_model - auc_base
cap_gain = cap_model - cap_base

BOLD = "My model predicts which pages will keep losing clicks, beats the old rule, and shows which titles we should rewrite first."

auc_clear = gain >= 0.02 and wins >= max(1, int(0.8 * nf))
if auc_clear and cap_gain >= 0:
    lead = f"{best_after} ranked pages better than the Week-4 rule, on AUC and in the top-5% queue"
elif auc_clear:
    lead = f"{best_after} ranked pages better than the Week-4 rule on AUC, but did not capture more later missed clicks in the top-5% review queue"
elif gain > 0 and wins > nf / 2:
    lead = f"{best_after} ranked pages slightly better than the Week-4 rule, but not clearly or repeatably"
else:
    lead = f"{best_after} did not rank pages better than the Week-4 rule"

SAFE = (f"In one sampled month (March 2026), on clients held out of training, {lead} "
        f"(observed AUC {auc_model:.2f} vs {auc_base:.2f}; better in {wins} of {nf} client folds; "
        f"top-5% queue captured {cap_model:.0%} vs {cap_base:.0%} of later missed clicks). "
        f"A simple impressions-only ranking reached AUC {imp_auc:.2f}" + (", so part of the signal is page size. " if imp_auc >= auc_base else " for comparison. ") +
        f"A random row split would have shown AUC {auc_naive:.3f} versus {auc_model:.3f} grouped" + (", a small difference here. " if (auc_naive - auc_model) < 0.02 else ", so the naive split flattered the model. ") +
        "The label is a proxy (later CTR gap versus a pooled norm), not a measured result of rewriting a title. "
        "This is directional decision-support for choosing what to review first, not evidence that a rewrite recovers clicks.")

print("BEFORE (too strong):\n ", BOLD)
print("\nAFTER (safe):\n ", SAFE)

BEFORE (too strong):
  My model predicts which pages will keep losing clicks, beats the old rule, and shows which titles we should rewrite first.

AFTER (safe):
  In one sampled month (March 2026), on clients held out of training, Gradient Boosting (small) ranked pages better than the Week-4 rule on AUC, but did not capture more later missed clicks in the top-5% review queue (observed AUC 0.91 vs 0.79; better in 5 of 5 client folds; top-5% queue captured 37% vs 39% of later missed clicks). A simple impressions-only ranking reached AUC 0.84, so part of the signal is page size. A random row split would have shown AUC 0.915 versus 0.906 grouped, a small difference here. The label is a proxy (later CTR gap versus a pooled norm), not a measured result of rewriting a title. This is directional decision-support for choosing what to review first, not evidence that a rewrite recovers clicks.


### More claims, rewritten

| Too strong | Safe version | Why it changed |
|---|---|---|
| "The model predicts clicks." | "The model ranks pages by a proxy for later missed clicks, observed on one month." | It predicts a proxy label, not clicks, and only for March 2026. |
| "Accuracy is high, so it works." | "AUC is measured on held-out clients; it is directional and below what a random split suggests." | Metric size depends on the split. |
| "These pages need a title rewrite." | "These pages are worth a human look first; low CTR can be normal for branded or SERP-feature queries." | The data cannot see query intent. |
| "Feature X drives clicks." | "Feature X was used by the model (permutation importance); this is not a causal statement." | Importance is not cause. |
| "No leakage." | "The audit checks listed above found no FLAGs in this sample; leakage cannot be ruled out in general." | A passed check is not proof. |

In [26]:
# ---- Receipt (OK to commit: counts, metrics and text only, no data rows) ----
os.makedirs("work/outputs", exist_ok=True)
out = {
    "assignment": "ML-09 validation audit",
    "split_before": f"KFold random rows, {n_splits} folds",
    "split_after": f"GroupKFold by client, {n_splits} folds, {n_clients} clients",
    "n_pages": int(len(M)),
    "metrics_before": s_before.round(4).to_dict(orient="index"),
    "metrics_after": s_after.round(4).to_dict(orient="index"),
    "inflation_before_minus_after": gap.to_dict(orient="index"),
    "best_model_after": best_after,
    "paired_auc_wins_vs_baseline": f"{wins}/{nf}",
    "pooled_norm_check": {"baseline_auc_pooled": round(float(b_pooled), 4), "baseline_auc_train_only": round(float(b_train), 4)},
    "label_derived_trap": {"clean_auc": round(auc_clean, 4), "trap_auc": round(auc_trap, 4)},
    "leakage_audit": A.to_dict(orient="records"),
    "claim_before": BOLD, "claim_after": SAFE,
}
with open("work/outputs/validation_audit.json", "w") as f:
    json.dump(out, f, indent=2, default=str)
print("Wrote work/outputs/validation_audit.json")

Wrote work/outputs/validation_audit.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it (Section 1 findings pasted from the paper, no `[...]` placeholders left)
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, tokens, or private queries anywhere (only shortened hashed ids appear)
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] The before/after table, the failure examples and the leakage audit summary are visible in the saved outputs
- [ ] I wrote 2-3 sentences of my own reading of the failure examples
- [ ] Committed to my repo under `work/notebooks/` (plus `work/outputs/validation_audit.json`), then submit the repo URL on the card. Done.